# Week 07: Content Action Playbook & Recommendation Engine

**Notebook:** `work/notebooks/w07_action_playbook.ipynb`  
**Goal:** Translate validated Week 06 ML scoring outputs into an actionable, human-reviewed SEO playbook. Generate a ranked queue with reason codes and export performance metrics/artifacts for paper synthesis.

---

## Section 1: Intended Use & Boundaries
* **Intended Use:** Directional decision-support tool for content teams to prioritize SEO optimization tasks (meta tag updates, layout refreshes, internal linking).
* **Out of Scope:** Autonomous direct publishing, automatic content generation without editorial review, or modifying high-converting landing pages.

In [1]:
import os
import json
import duckdb
import pandas as pd
import numpy as np

# Ensure target output folders exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# 1. Synthesize Scored Performance Dataset
np.random.seed(42)
n_samples = 50

urls = [f'https://flyrank.com/blog/article-{i}' for i in range(1, n_samples + 1)]
impressions = np.random.randint(800, 10000, size=n_samples)
ctr = np.random.uniform(0.005, 0.04, size=n_samples)
position = np.random.uniform(1.5, 18.0, size=n_samples)
model_score = np.random.uniform(0.30, 0.98, size=n_samples)

df = pd.DataFrame({
    'url': urls,
    'impressions': impressions,
    'ctr': ctr,
    'position': position,
    'model_score': model_score
})

# 2. Archetype Mapping & Reason Code Assignment
def assign_action(row):
    if row['model_score'] >= 0.75 and row['ctr'] < 0.018 and row['position'] <= 10:
        return 'REASON_CTR_GAP', 'Title & Meta Description Optimization', 'HIGH'
    elif row['model_score'] >= 0.60 and row['position'] > 10:
        return 'REASON_RANK_DECAY', 'Content Refresh & Internal Linking Boost', 'MEDIUM'
    elif row['impressions'] > 5000 and row['ctr'] < 0.015:
        return 'REASON_HIGH_IMP_LOW_CTR', 'UX / Snippet Schema Refresh', 'HIGH'
    else:
        return 'REASON_MAINTAIN', 'Monitor Performance (No Immediate Action)', 'LOW'

actions = [assign_action(row) for _, row in df.iterrows()]
df['reason_code'] = [a[0] for a in actions]
df['recommended_action'] = [a[1] for a in actions]
df['priority'] = [a[2] for a in actions]

# Sort Action Queue by Priority Score
ranked_queue = df.sort_values(by=['model_score', 'impressions'], ascending=[False, False]).reset_index(drop=True)

print("--- TOP 5 RANKED ACTIONABLE RECOMMENDATIONS ---")
print(ranked_queue[['url', 'priority', 'reason_code', 'recommended_action', 'model_score']].head().to_string(index=False))

# 3. Export Queue CSV & Metrics JSON (Work/Outputs Artifacts)
ranked_queue.to_csv('../outputs/ranked_action_queue.csv', index=False)

metrics_summary = {
    "total_urls_evaluated": int(len(df)),
    "high_priority_actions": int((df['priority'] == 'HIGH').sum()),
    "medium_priority_actions": int((df['priority'] == 'MEDIUM').sum()),
    "low_priority_actions": int((df['priority'] == 'LOW').sum()),
    "top_reason_code": str(df['reason_code'].mode()[0]),
    "export_timestamp": "2026-08-15"
}

with open('../outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)

print("\n Exports generated successfully under work/outputs/")

--- TOP 5 RANKED ACTIONABLE RECOMMENDATIONS ---
                                url priority       reason_code                        recommended_action  model_score
https://flyrank.com/blog/article-11   MEDIUM REASON_RANK_DECAY  Content Refresh & Internal Linking Boost     0.970242
https://flyrank.com/blog/article-35     HIGH    REASON_CTR_GAP     Title & Meta Description Optimization     0.936976
https://flyrank.com/blog/article-39     HIGH    REASON_CTR_GAP     Title & Meta Description Optimization     0.928792
 https://flyrank.com/blog/article-7      LOW   REASON_MAINTAIN Monitor Performance (No Immediate Action)     0.917621
https://flyrank.com/blog/article-49      LOW   REASON_MAINTAIN Monitor Performance (No Immediate Action)     0.912284

 Exports generated successfully under work/outputs/


## Section 3: Human Review Protocol & Automation Guardrails

### 🚫 The "NO-GO" Automation List
1. **Direct Production CMS Writes:** Zero AI-suggested content updates or meta tags may publish autonomously without human editor verification.
2. **Legal & Financial Routes:** Privacy policy, terms, or monetization disclosure pages are strictly excluded from model evaluation.
3. **High-Value Conversion Pages:** Landing pages driving $>10\%$ of revenue require manual sign-off regardless of model priority score.

### 📋 Human Review Checklist
- [ ] Confirm suggested title tags adhere to brand tone and character constraints ($<60$ chars).
- [ ] Ensure keywords do not cannibalize sibling URLs.
- [ ] Verify seasonal trends before declaring rank decay.

## Section 4: Monitoring Triggers & Self-Check

| Trigger Event | Condition | Action |
| :--- | :--- | :--- |
| **Data Drift** | Average CTR shifts by $>15\%$ | Re-scale feature preprocessing pipeline |
| **Search Update Alert** | Major search algorithm update detected | Pause action queue for 14 days |
| **Concept Drift** | Model ROC-AUC drops below $0.80$ | Retrain pipeline on latest 60-day window |

---

## Section 5: Self-Check Checklist
- [x] Defined intended use, limits, and explicit non-production guardrails.
- [x] Categorized reason codes and mapped archetypes to clear actions.
- [x] Established explicit "No-Go" cases for automation.
- [x] Generated and exported `work/outputs/ranked_action_queue.csv` and `work/outputs/playbook_metrics.json`.



---

# Week 8 Showcase: 5-Minute Demo + Shareable Cuts

## 5-Minute Demo Outline

**0:00–0:45 — Question**
- FlyRank content teams can have more candidate pages to review than they can prioritize manually.
- Working question: can observable search-performance signals rank pages that may deserve SEO review without leaking outcome information?

**0:45–1:45 — Method**
- Framed the task as binary classification for CTR opportunity / underperformance prioritization.
- Compared a rule-based baseline with a constrained Random Forest.
- Added explicit leakage checks and a time-aware validation audit.
- Kept the output as a human-reviewed action queue rather than an autonomous publishing system.

**1:45–2:45 — One chart**
- Show the model-vs-baseline chart from the paper.
- Point out that the Random Forest has higher ROC-AUC in the supplied synthetic fixture (0.731 vs 0.677), while the rule baseline is stronger on accuracy, precision, and recall.
- Main lesson: complexity alone is not evidence that the model is better for the operating objective.

**2:45–3:45 — One honest result**
- The leakage experiment produced ROC-AUC 0.9395 with a label-derived feature and 0.5072 after that leaked feature was removed.
- This demonstrates why feature provenance matters and why the 0.9395 result must not be presented as legitimate model performance.

**3:45–4:45 — One recommendation**
- Use the score as a review queue: prioritize high-impression / low-CTR candidates for editor inspection, then map the issue to title/meta, content/internal-linking, or SERP/UX review.

**4:45–5:00 — Close**
- Re-state the boundary: this is directional decision support, not proof of causal lift or a production-ready autonomous optimizer.

## Shareable Cut 1 — Short Social Post

I built an audited ML prototype for SEO prioritization: a rule-based baseline vs. a constrained Random Forest, with explicit leakage checks and time-aware validation. The most useful result was not a flashy score—it was seeing how a label-derived feature pushed ROC-AUC to 0.9395, then watching it fall to 0.5072 once the leak was removed. The takeaway: for search analytics, trustworthy feature provenance and realistic validation matter as much as model choice.

## Shareable Cut 2 — Employer-Facing Summary

I built a public-safe ML decision-support prototype for prioritizing SEO content reviews, using search-performance signals such as impressions, CTR, position, and content age. I evaluated a rule-based baseline against a constrained Random Forest and audited the workflow for leakage and time-aware generalization. On the supplied synthetic fixtures, the model improved ROC-AUC over the baseline but did not win every metric, reinforcing an evidence-first approach to model selection and human-reviewed SEO recommendations.
